# Ejercicio 1.


#### _Respuesta inciso a_.
##### ¿Cual es el rango de $Q\in \mathbb{R}^{2n \times 12}$ para que la solución sea no nula única a menos de un factor de escala?
Para obtener una solución no nula $\text{dim}(\text{Nu}(Q)) \geq 1$ y para que sea a un factor de escala entonces $\text{Nu}(Q) = 1$, si $Q$ es la matriz de una transformación lineal $T$ entonces $T: \mathbb{V} \to \mathbb{W}$ donde $\mathbb{V} = \mathbb{R}^{12}$ y $\text{Im}(T) \subset \mathbb{R}^{2n}$. Por el teorema de la dimensión: $\text{dim}(\text{Im}(Q))+ \text{dim}(\text{Nu}(Q)) = \text{dim}(\text{Im}(Q)) + 1 = 12 \iff \text{dim}(\text{Im}(Q)) = 11$. Por lo tanto, como $\text{dim}(\text{Im}(Q)) = \text{rg}(Q)$, se tiene que el rango de $Q$ es igual a 11.
##### ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?
Teniendo en cuenta que cada punto aporta dos ecuaciones lineales independientes por $\mathbb{V} = \mathbb{R}^{2n}$ con $n \in \mathbb{N}$ como el número de puntos y que $2n \geq \text{rg}(Q)$, entonces $2n \geq 11 \implies n=6$. Por lo tanto, el número mínimo de puntos necesarios que aseguran la solución son 6.
La condición que deben cumplir es que una de las 12 filas de $Q$ debe ser combinación lineal de las otras 11 porque si $\text{rg}(Q)=12 \implies \text{Nu}(Q) = 0 \implies m = \vec{0}$, luego la matriz $M$ es nula. Si $\text{rg}(Q) \leq 10$ entonces hay puntos que pertenecen al mismo plano y $\text{Nu}(Q) \geq 2 $ provocando infinitas soluciones. Por ejemplo, si $\text{Nu}(Q) = 2 $ entonces existen $m_1, m_2$ con matrices asociadas $M_1, M_2$ y luego $M = Q(\lambda_1 m_1 + \lambda_2 m_2)$.
En conclusión 11 de las 12 filas de $Q$ son linealmente independientes y los puntos no pueden pertenecer a un mismo plano, pues entonces habría dependencia lineal, lo cual disminuiría el rango de la matriz y en consucuencia aumentando la dimensión de su núcleo obteniendo así infinitas soluciones.

#### _Respuesta inciso b_.
En teoría deberíamos hallar la solucion del sistema $Qm = 0$ donde $rg(Q) = 11$. Pero en la práctica, generalmente los errores podrían provocar que $rg(Q) = 12$, obteniendo la solución trivial y la matriz $M$ resultaría nula. Para evitar obtener la solución trivial se busca resolver el problema de mínimos cuadrados homogéneos con restricción de norma.
Toda matriz rectangular posee descomposición en valores singulares por lo que $Q = U \Sigma V^t$. Como $U$ es unitaria se tiene que:
$$
\begin{aligned}
Qm &= U \Sigma V^t m \\
&=  U (\Sigma V^t m) \\
&= \Sigma V^t m
\end{aligned}
$$
Entonces $||Qm||_2 ^2 = ||\Sigma V^t m||_2 ^2$.
Como $V$ es unitaria, tomando como vectores sus columnas, estos forman una base ortonormal de $\mathbb{R}^{12}$ de modo que $m = \sum_{i=1}^{12}\alpha_i v_i$ con $\sum_{i=1}^{12}\alpha_i^2 = 1$. Sea $A = (\alpha_1, \alpha_2, ... , \alpha_{12}) \in \mathbb{R}^{12}$, de forma matricial $m = VA$, luego
$$
\begin{aligned}
\Sigma V^t m &= \Sigma V^t VA \\
&= \Sigma (V^t V)A \\
&= \Sigma A
\end{aligned}
$$
De esta manera,
$$
\begin{aligned}
||Qm||_2 ^2 &= ||\Sigma A||_2 ^2 \\
&= \sum_{i=1}^{12} \sigma_i^2 \alpha_i^2 \\
&\geq \sigma_{12}^2 \sum_{i=1}^{12} \alpha_i^2 = \sigma_{12}^2
\end{aligned}
$$
El valor mínimo de la norma es $\sigma_{12}^{2}$. Este mínimo se alcanza cuando $A=\pm(0, 0, ... , 1)$ y por lo tanto $m = \pm v_{12}$.

# Ejercicio 2.

## Descripción de los Datasets.
### `detected_corners.txt`
El archivo contiene una tabla de `float` separados por espacios de dimensión $(210 \times 24)$.

Cada fila representa una imagen que contiene las coordenadas 2D de 12 puntos detectados. Las columnas están agrupadas en pares $(u_1, v_1), (u_2, v_2), \dots, (u_12, v_12)$.

### `p_W_corners.txt`
Contiene una tabla de `float` separados por comas ',' de dimension $(12 \times 3)$.

Cada i-ésima fila contiene las coordenadas 3D asociadas a la i-ésima coordenada 2D de una imagen cualquiera en `detected_corners.txt`.

In [1]:
import numpy as np
from alc import *
np.set_printoptions(precision=8, suppress=True, linewidth=200)
ruta = 'TP/template-alumnos-20261021/calib/3d/'
fotos = np.loadtxt(fname=ruta+'detected_corners.txt') # pixels (2D)
pcorners = np.loadtxt(fname=ruta+'p_W_corners.txt',delimiter=',') # dimensiones 3D

fotos.shape

(210, 24)

In [2]:
objPts = convertirP(pcorners)
uvp = puntosfotoi(20,fotos)
uv_norm, P_norm, T_img, T_world = normalizacionDePuntos(uvp, objPts)
print(P_norm.shape)
Q_nor = obtenerQ3d(uv_norm, P_norm)

(12, 4)


## Obtencion de $M$, $K$ y $t$
Para obtener la descomposicion de R, puedo utilizar la descomposicion RQ que sale de hacer la inversa de su QR.
Para ello necesito las funciones calculaQR y la inversa, (mi inversa usa la descomposicion LU de la matriz).
$M = [A|b]$ donde $M = [KR|KT]$, $A = KR$ y $b = Kt$.

Separando `M[:,0:3] = A` y `M[:,3] = b`

In [3]:
Mhat = resolverDLT3D(Q_nor)
print(Mhat.shape)
M = multMat(inversa(T_img), multMat(Mhat, T_world))
A = M[:,0:3].copy()
b = M[:,3].copy()
# Como quiero sacar RQ en vez de QR calculo la inversa de A y luego la descomp QR me termina quedando KR
Ainv = inversa(A)
Qinv, Rinv = QR_con_HH(Ainv)
K = inversa(Rinv)
R = Qinv.T

# K[2,2] es el factor de escala, debo dividir a b y K por este valor
Knormalizada = K / K[2, 2]


(3, 4)


In [4]:
# Como b = Kt entonces t = Kinv * B
t = multMat(inversa(K),b.reshape(-1,1))
t = t.flatten() # es lo mismo que reshape(1,-1)
# no uso la Knormalizada pues si ponele que esta escalada por un valor $, entonces mi K sin normalizar seria mas un $K, y cuando hago la inversa me queda 1/$ * Kinv. entonces al multiplicar por por $b ya estaria normalizando
# De la DLT obtenemos la matriz con una escala global $ (osea se pone lambda pero ni idea como ponerlo aca xd)


In [5]:
Mhat = resolverDLT3D(Q_nor)
M = multMat(inversa(T_img), multMat(Mhat, T_world))
R,t,K = devolverMatricesCalibracion3d(M,k12igual0=True,uigualv=True)

## Cuantificacion de los errores


Error aplicando normalización isotrópica.

In [6]:
# ls = []
# for i in range(len(fotos)):
#     imgPts = puntosfotoi(i, fotos)
#     Q=obtenerQ3d(imgPts , objPts)
#     M=resolverDLT3D(Q)
#     err = errorRelativoMatrizProyeccion(M, imgPts, objPts)

#     imgPtsNor, objPtsNor, T_img, T_world = normalizacionDePuntos(imgPts, objPts)
#     Qhat = obtenerQ3d(imgPtsNor, objPtsNor)
#     Mhat = resolverDLT3D(Qhat)
#     Mnor = multMat(inversa(T_img), multMat(Mhat, T_world))
#     err_nor = errorRelativoMatrizProyeccion(M, imgPts, objPts)

#     t = (err, err_nor)
#     ls.append(t)
# print(max(ls, key=lambda x: x[0]))


No se hallaron diferencias significativas entre normalizar puntos y no hacerlo para las fotos dadas.

# Ejercicio 3: Calibración de puntos coplanares

### ¿Cuál es el rango de $Q \in \mathbb{R}^{2n \times 9}$ para asegurar una solución única a menos de un factor de escala?

Para obtener una solución no nula entonces $\text{dim}(\text{Nu}(Q)) \geq 1$ y para que sea a un factor de escala entonces $\text{Nu}(Q) = 1$, si $Q$ es la matriz de una transformación lineal $T: \mathbb{V} \to \mathbb{W}$ donde $\mathbb{V} = \mathbb{R}^{9}$ y $\text{Im}(T) \subset \mathbb{R}^{2n}$. Por el teorema de la dimensión: $\text{dim}(\text{Im}(Q))+ \text{dim}(\text{Nu}(Q)) = \text{dim}(\text{Im}(Q)) + 1 = 9 \iff \text{dim}(\text{Im}(Q)) = 8$. Por lo tanto, como $\text{dim}(\text{Im}(Q)) = \text{rg}(Q)$, se tiene que el rango de $Q$ es igual a 8. 

## Comprobaciones de herramientas implementadas

In [7]:
import cv2
#ruta = 'TP/template-alumnos-20261021/calib/coplanar/'
filas = 7  # j = 0, ..., 6
columnas = 11  # i = 0, ..., 10
tam_cuadrado = 30  # 30 mm
objp = np.zeros((7*11,3)) # creo mis puntos donde voy a guardar las esquinas, y mi punto 3d
idx = 0 # id de 
for i in range(0,filas):
  for j in range(0,columnas):
    objp[idx,0] = j * tam_cuadrado# Eje X
    objp[idx,1] = i * tam_cuadrado# Eje Y
    objp[idx,2] = 1 # osea pues mi Z es 0 siempre para el plano
    # Porque las columnas son mi X y mi filas son mi Y, bueno si miramos como se mueven en las matrices tiene sentido las filas van de arriba a abajo y las columnas de izquierda a derecha.
    idx+=1 # esto va a llegar hasta el final tipo 77
rutas_imagenes = []
for i in range(1,11):
  rutas_imagenes.append(f'TP/template-alumnos-20261021/calib/coplanar/Im_R_{i}.png') # Iteramos todas las rutas de las fotos
rutas_imagenes
puntos3D = []
puntos2D = []



In [8]:
for ruta in rutas_imagenes:
  img = cv2.imread(ruta) # Leo las rutas, osea leo cada imagen
  imgGray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) # esto convierte a escala de grises los datos pues findChessboardCorners usa escala de grises no puedo meter a color

  encontrado, corners = cv2.findChessboardCorners(imgGray , (columnas,filas), None) # (imagenGrises,(7,11) flags= None osea por defecto todo)
  # devuelve si encontro o no con True o False, y en corners me devuelve mis u,v que necesito para los puntos.
  if encontrado == True:
    # Si tengo el punto encontrado, lo combino con mi punto de objp
    puntos3D.append(objp)
    puntos2D.append(corners)
    print(f"Tablero detectado correctamente en: {ruta}")
  else:
    print(f"No se pudo detectar el tablero en {ruta}")

Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_1.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_2.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_3.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_4.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_5.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_6.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_7.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_8.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_9.png
Tablero detectado correctamente en: TP/template-alumnos-20261021/calib/coplanar/Im_R_10.png


In [9]:
# Probamos con la primera imagen para generar su Q
puntos2DLimpios = []
for i in range(0, len(puntos2D)):
    puntos2DLimpios.append(np.array(puntos2D[i]).reshape(-1,2))
puntos2D = np.array(puntos2DLimpios)
Q = obtenerQ2d(puntos2D[0], puntos3D[0])
print(Q.shape)

(154, 9)


Probamos armarA. Empezamos armando Hlst.

In [10]:
Hlist = [resolverQh(obtenerQ2d(puntos2D[i], puntos3D[i])) for i in range(len(puntos2D))]
A = armarA(Hlist)
print(A)

[[-0.00000025  0.00001261  0.00000029 -0.          0.         -0.        ]
 [ 0.0000127   0.00000107 -0.00001251  0.          0.          0.        ]
 [ 0.00000229  0.00000951 -0.00000218  0.          0.         -0.        ]
 [ 0.00000955 -0.00000894 -0.00000947  0.         -0.          0.        ]
 [ 0.0000007   0.00001195 -0.0000005   0.          0.          0.        ]
 [ 0.00001208 -0.0000024  -0.00001182  0.         -0.          0.        ]
 [ 0.00000057  0.00001174 -0.00000004  0.          0.          0.        ]
 [ 0.00001162 -0.00000123 -0.00001183  0.         -0.         -0.        ]
 [-0.00000094  0.00001128  0.00000107 -0.          0.         -0.        ]
 [ 0.00001177  0.00000402 -0.00001081  0.          0.          0.        ]
 [-0.00000031  0.00001161  0.00000052 -0.          0.         -0.        ]
 [ 0.000012    0.00000169 -0.00001122  0.          0.          0.        ]
 [ 0.00000082  0.00001102 -0.0000003   0.          0.          0.        ]
 [ 0.00001079 -0.00000227

## Pruebo la busqueda de los errores normalizando

In [11]:
ls = []
for i in range(len(puntos2D)):
    uvs = puntos2D[i]
    Ps = puntos3D[i]
    Q = obtenerQ2d(uvs, Ps)
    H = resolverQh(Q)
    err = errorRelativoMatrizProyeccion(H, uvs, Ps)
    uvs_nor, Ps_nor, T_img, T_world = normalizacionDePuntos(uvs, Ps, homografia=True)
    Qn = obtenerQ2d(uvs_nor, Ps_nor)
    Hn = resolverQh(Qn)
    Hn = multMat(inversa(T_img), multMat(Hn, T_world))
    err_n = errorRelativoMatrizProyeccion(Hn, uvs, Ps)
    ls.append((err, err_n))
print(max(ls, key=lambda x: x[0]))

d:\Multimedia\docs\Materias carrera\Algebra Lineal Computacional\TP-MMUA\alc.py:998: RuntimeWarning: divide by zero encountered in divide
  uv_obtenido = uv_obtenido[:2]/uv_obtenido[2]


(np.float64(0.00018235107717959619), np.float64(4.064514201267576e+31))


# Ejercicio 5